In [1]:
!pip install weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.7/603.7 kB 9.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.0 MB/s eta 0:00:00


In [7]:
import os
import shutil
import weaviate
import pandas as pd
import random
from tqdm import tqdm
from typing import List, Dict, Union
from sentence_transformers import SentenceTransformer

In [8]:
source_db_path = '/kaggle/input/vecrtordb/kaggle/working/vectordb'
writable_db_path = '/kaggle/working/vectordb_copy'

if os.path.exists(source_db_path):
    if os.path.exists(writable_db_path):
        shutil.rmtree(writable_db_path)
    
    print("Copying vector database to writable directory...")
    shutil.copytree(source_db_path, writable_db_path)
    print("Copy complete.")
else:
    print(f"Error: Source path {source_db_path} not found.")

Copying vector database to writable directory...
Copy complete.


In [15]:
def calculate_hybrid_accuracy(
    collection, 
    embedder, 
    test_df: pd.DataFrame, 
    k_values: List[int] = [1, 3, 5], 
    alpha: float = 0.7
) -> Dict[int, float]:
    
    max_k = max(k_values)
    
    correct_matches = {k: 0 for k in k_values}
    total_samples = len(test_df)
    
    print(f"Running Hybrid Search (alpha={alpha}) on {total_samples} samples...")
    print(f"Calculating metrics for: {', '.join([f'Top-{k}' for k in k_values])}")

    for _, row in tqdm(test_df.iterrows(), total=total_samples):
        query_text = row['text'].lower()
        target_song = row['song'].lower()
        target_artist = row['artist'].lower()
        
        query_vector = embedder.encode(query_text).tolist()
        
        response = collection.query.hybrid(
            query=query_text,
            vector=query_vector,
            alpha=alpha,
            limit=max_k
        )
        
        found_rank = -1 
        
        for rank, obj in enumerate(response.objects):
            if (obj.properties['song'] == target_song and 
                obj.properties['artist'] == target_artist):
                found_rank = rank + 1 
                break
        
        if found_rank != -1:
            for k in k_values:
                if found_rank <= k:
                    correct_matches[k] += 1

    results = {}
    for k in k_values:
        accuracy = (correct_matches[k] / total_samples) * 100
        results[k] = round(accuracy, 2)
        
    return results

In [4]:
df = pd.read_csv('/kaggle/input/57651-spotify-songs/Spotify Million Song Dataset_exported.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
client = weaviate.connect_to_embedded(
    version="latest",
    persistence_data_path=writable_db_path, 
    environment_variables={"LOG_LEVEL": "error"}
)

print(f"Connected: {client.is_live()}")
print(f"Collections: {list(client.collections.list_all().keys())}")

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 159
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Connected: True
Collections: ['Spotify_songs']


{"action":"hnsw_load_commit_log_corruption","build_git_commit":"","build_go_version":"go1.25.5","build_image_tag":"","build_wv_version":"1.35.2","level":"error","msg":"write-ahead-log ended abruptly, some elements may not have been recovered","path":"/kaggle/working/vectordb_copy/spotify_songs/d4U6wzmPORh2/vectors_default.hnsw.commitlog.d/1767504711","time":"2026-01-04T05:57:41Z"}


In [10]:
collection = client.collections.get("Spotify_songs")

In [11]:
embedder = SentenceTransformer("ibm-granite/granite-embedding-small-english-r2")

modules.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/95.3M [00:00<?, ?B/s]

sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7817f1959e10>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7817f1959860>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7817f195a120>
sys:1: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7817f19599b0>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [16]:
test_samples = df.sample(1000, random_state=42)

accuracies = calculate_hybrid_accuracy(
    collection=collection,
    embedder=embedder,
    test_df=test_samples,
    k_values=[1, 3, 5],
    alpha=0.7
)

print("\nFinal Results:")
for k, acc in accuracies.items():
    print(f"Top-{k} Accuracy: {acc}%")

Running Hybrid Search (alpha=0.7) on 1000 samples...
Calculating metrics for: Top-1, Top-3, Top-5


100%|██████████| 1000/1000 [06:30<00:00,  2.56it/s]


Final Results:
Top-1 Accuracy: 99.1%
Top-3 Accuracy: 99.9%
Top-5 Accuracy: 100.0%
